In [ ]:
import pandas as pd
import os
import multiprocessing
import functions_stanza
import pickle

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
dfs = []

for file in os.listdir("gerunzius_download"):
    csv = pd.read_csv(f"gerunzius_download/{file}", sep=";", header=None)
    csv = csv.set_index(csv[0])
    csv = csv.drop(columns=[0])
    dfs.append(csv)

In [ ]:
df = pd.concat(dfs)

In [ ]:
df = df.sort_index()

In [ ]:
df = df.drop_duplicates().reset_index(drop=True) # remove duplicates; export from kontext pretty iffy

In [ ]:
df = df.fillna("")

In [ ]:
df[3] = df[3].str.replace("-", "")

In [ ]:
df["romanian"] = df[2] + " " + df[3] + " " + df[4]

In [ ]:
replacements = [
    ("‘ ", "‘"),
    (" ’", "’"),
    (" ,", ","),
    (" .", "."),
    (" :", ":"),
    (" ?", "?"),
    ("( ", "("),
    (" )", ")"),
    (" ;", ";"),
    ("' ", "'"),
    (" '", "'"),
    (" -", "-"),
    ("- ", "-"),
    ("ş", "ș"),
    ("ţ", "ț"),
]

In [ ]:
for replacement in replacements:
    df["romanian"] = df["romanian"].str.replace(replacement[0], replacement[1])

In [ ]:
for replacement in replacements:
    df[7] = df[7].str.replace(replacement[0], replacement[1])

In [ ]:
for replacement in replacements:
    df[3] = df[3].str.replace(replacement[0], replacement[1])

In [ ]:
CHUNK_SIZE = 2500

# Parse English

In [ ]:
multiprocessing.set_start_method("spawn", force=True)

In [ ]:
process_en = multiprocessing.Process(
    target=functions_stanza.parse_in_chunks, args=(df, "en", 7, "gerunzius", CHUNK_SIZE)
)

In [ ]:
process_en.start()

In [ ]:
parsed_dict_en = functions_stanza.load_pickled_data(folder="gerunzius", language="en")

In [ ]:
parsed_dict_en = pickle.load(open("gerunzius/parsed_dict_en.pkl", "rb"))

In [ ]:
adv_part_indices_and_words = []

In [ ]:
for i in range(0, len(parsed_dict_en)):
    adv_parts = functions_stanza.get_adv_part_in_sentences(parsed_dict_en[i])
    if adv_parts:
        adv_part_indices_and_words.append((i, adv_parts))

In [ ]:
len(adv_part_indices_and_words)

In [ ]:
adv_part_indices_and_words

# Parse Romanian

In [ ]:
multiprocessing.set_start_method("spawn", force=True)

In [ ]:
process_ro = multiprocessing.Process(
    target=functions_stanza.parse_in_chunks,
    args=(df, "ro", "romanian", "gerunzius", CHUNK_SIZE),
)

In [ ]:
process_ro.start()

In [ ]:
multiprocessing.active_children()

In [ ]:
parsed_dict_ro = functions_stanza.load_pickled_data(folder="gerunzius", language="ro")

In [ ]:
parsed_dict_ro = pickle.load(open("gerunzius/parsed_dict_ro.pkl", "rb"))

In [ ]:
gerunziu_indices_and_lemmas = []

In [ ]:
for i in range(0, len(df)):
    row = df.iloc[i]
    contains_gerunziu, lemma_of_gerunziu = functions_stanza.sentences_contain_gerunziu(
        row[3], parsed_dict_ro[i]
    )
    if contains_gerunziu:
        gerunziu_indices_and_lemmas.append((i, lemma_of_gerunziu))

In [ ]:
gerunziu_indices_and_lemmas

# Building combined df

In [ ]:
gerunziu_indices_and_lemmas_df = pd.DataFrame(
    gerunziu_indices_and_lemmas, columns=["index", "lemma"]
).set_index("index")

In [ ]:
adv_part_indices_and_words_df = pd.DataFrame(
    adv_part_indices_and_words, columns=["index", "adv_parts_and_lemmas"]
).set_index("index")

In [ ]:
gerunziu_indices_and_lemmas_df

In [ ]:
adv_part_indices_and_words_df

In [ ]:
df = pd.concat(
    [df, gerunziu_indices_and_lemmas_df, adv_part_indices_and_words_df],
    axis=1,
)

In [ ]:
df["is_gerunziu"] = False
df.loc[~df["lemma"].isna(), "is_gerunziu"] = True

In [ ]:
df.head(2)

In [ ]:
df["has_adv_participle"] = False
df.loc[~df["adv_parts_and_lemmas"].isna(), "has_adv_participle"] = True

In [ ]:
len(df.loc[df["is_gerunziu"]])

In [ ]:
df.to_csv("extraction_output/annotated_all_ger_to_part.csv")

In [ ]:
df.loc[df["is_gerunziu"] & df["has_adv_participle"]].to_csv(
    "extraction_output/gerunziu_with_adv_part.csv"
)

# Assigning gerunzius to adverbial participles

In [ ]:
import functions_simalign

In [ ]:
df = pd.read_csv("extraction_output/gerunziu_with_adv_part.csv", index_col=0)

In [ ]:
df["3"] = df["3"].str.replace("ş", "ș")
df["3"] = df["3"].str.replace("ţ", "ț")

In [ ]:
multiprocessing.set_start_method("spawn", force=True)

In [ ]:
process_matching = multiprocessing.Process(
    target=functions_simalign.match_adv_part_and_gerunzius, args=(df, "ro")
)

In [ ]:
process_matching.start()

In [ ]:
multiprocessing.active_children()

In [ ]:
df.to_csv("extraction_output/ger_adv_matches.csv")